In [3]:
from typing import List, Dict
import ujson as json
from pathlib import Path
from openai import OpenAI

# Self-ask / self-instruct
# Self-Ask + filter
# DeepEval Synthetic Module
# Langchain SyntheticQAEvaluator

In [4]:
# common interface

QA = Dict[str, str]  # {"question": ..., "answer": ..., "doc_id": ..., "source_text": ...}

def generate_qas_for_doc(doc_id: str, text: str, framework: str) -> List[QA]:
    ...


# for doc in arxiv_docs:
#     qas = generate_qas_for_doc(doc.id, doc.abstract_or_section, framework="self_instruct")
#     # or framework="langchain", "deepeval", etc.


In [36]:
# turning chunks to slices

CHUNK_DIR = Path("data/rag_chunks_v2")

def load_chunks(path: Path) -> List[Dict]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)  # your file seems to be a JSON list



def make_slices_from_chunks(
    chunks: List[Dict],
    max_chars: int = 5000,
    min_chars: int = 2000,
) -> List[Dict]:
    """
    Returns list of slices:
    [
      {"doc_id": ..., "slice_id": ..., "text": ..., "chunk_ids": [...]},
      ...
    ]
    """
    slices = []
    cur_text = []
    cur_len = 0
    cur_chunk_ids = []

    # assume chunks are already in reading order
    for ch in chunks:
        if ch.get("type") != "paragraph":
            continue
        t = (ch.get("content") or "").strip()
        if not t:
            continue

        if cur_len + len(t) > max_chars and cur_len >= min_chars:
            # close current slice
            slice_idx = len(slices)
            doc_id = ch["id"].split("_")[0]   # adjust if you store doc_id elsewhere
            slices.append({
                "doc_id": doc_id,
                "slice_id": f"{doc_id}_slice_{slice_idx}",
                "text": "\n\n".join(cur_text),
                "chunk_ids": cur_chunk_ids,
            })
            cur_text, cur_len, cur_chunk_ids = [], 0, []

        # add to current slice
        cur_text.append(t)
        cur_len += len(t)
        cur_chunk_ids.append(ch["id"])

    # flush last slice
    if cur_text:
        doc_id = chunks[0]["id"].split("_")[0]
        slice_idx = len(slices)
        slices.append({
            "doc_id": doc_id,
            "slice_id": f"{doc_id}_slice_{slice_idx}",
            "text": "\n\n".join(cur_text),
            "chunk_ids": cur_chunk_ids,
        })

    return slices

In [37]:
# API Calls to LLM

from dotenv import load_dotenv

load_dotenv("keys.env") 
client = OpenAI()
MODEL = "gpt-4.1-mini"  


def get_response(prompt: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[{"role": "user",
                "content": [{"type": "input_text", "text": prompt}]}],
    )
    return resp.output_text.strip()


In [52]:
# --- helper to safely extract JSON ---
def extract_json(raw: str) -> dict:
    raw = raw.strip()

    # Strip ``` fences if present
    if raw.startswith("```"):
        lines = raw.splitlines()
        lines = [ln for ln in lines if not ln.strip().startswith("```")]
        raw = "\n".join(lines).strip()

    # First try: as-is
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Second try: escape backslashes (for \alpha, \nabla, etc.)
    try:
        fixed = raw.replace("\\", "\\\\")
        return json.loads(fixed)
    except json.JSONDecodeError as e:
        print("JSON parse error even after fixing backslashes:", e)
        print("Raw output:\n", raw)
        # Fallback: return empty structure so rest of pipeline doesn't crash
        return {"qas": []}

# self-ask / self-instruct

In [39]:
path = CHUNK_DIR / "2211.17192.rag.chunks.json"
chunks = load_chunks(path)
slices = make_slices_from_chunks(chunks)

print(len(slices), "slices")
print(slices[0]["text"]) 

9 slices
Given the importance of large autoregressive models and specifically large Transformers, several approaches were

Speculative execution (Burton, 1985; Hennessy & Patterson, 2012) is an optimization technique, common in processors, where a task is performed in parallel to verifying if it's actually needed - the payoff being increased concurrency. A well-known example of speculative execution is branch prediction. For speculative execution to be effective, we need an efficient mechanism to suggest tasks to execute that are likely to be needed. In this work, we generalize speculative execution to the stochastic setting - where a task might be needed with some probability. Applying this to decoding from autoregressive models like Transformers, we sample generations from more efficient approximation models as speculative prefixes for the slower target models . With a novel sampling method, speculative sampling , we maximize the probability of these speculative tasks to

[START] jap

In [40]:
self-ask / self-instruct


SELF_INSTRUCT_PROMPT = """
You are a helpful assistant reading a research paper excerpt.

TEXT:
\"\"\"{text}\"\"\"

Generate {n_qas} diverse question-answer pairs that can be answered *directly and unambiguously* from this text alone.

Requirements:
- Cover different types: definitions, methods, motivations, comparisons, results.
- Make questions specific, not vague.
- Answers should be concise and copy or paraphrase the text.
- Return JSON as a list under key "qas", like:
  {{"qas": [{{"question": "...", "answer": "..."}}, ...]}}
"""

In [53]:

def generate_qas_self_instruct(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    
    prompt = SELF_INSTRUCT_PROMPT.format(text=text, n_qas=n_qas)
    response = get_response(prompt)
    data = extract_json(response)

    qas: List[QA] = []
    for qa in data.get("qas", []):
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas

In [54]:
slice0 = slices[0]
qas = generate_qas_self_instruct(
    doc_id=slice0["doc_id"],
    text=slice0["text"],
    n_qas=5,
)

for qa in qas:
    print("Q:", qa["question"])
    print("A:", qa["answer"])
    print("----")

Q: What is speculative execution and where is it commonly used?
A: Speculative execution is an optimization technique, common in processors, where a task is performed in parallel to verifying if it's actually needed, with the payoff being increased concurrency.
----
Q: How does the paper generalize speculative execution for autoregressive models like Transformers?
A: The paper generalizes speculative execution to the stochastic setting by sampling generations from more efficient approximation models as speculative prefixes for the slower target models using a novel sampling method called speculative sampling.
----
Q: What are the main contributions of this work?
A: (1) A generalization of speculative execution to the stochastic setting, with a novel sampling method called speculative sampling, and (2) A decoding mechanism called speculative decoding that accelerates decoding from autoregressive models without changing model architectures, training regimes, or output distributions.
----

## generating and storing dataset using self-ask

In [55]:
OUT_PATH = Path("eval_qas_self_instruct.jsonl")
DATASET_NAME = "arxiv_scale_rag_self_instruct_v1" 

In [56]:
from datetime import datetime


def build_eval_qas_jsonl(
    chunk_dir: Path = CHUNK_DIR,
    out_path: Path = OUT_PATH,
    n_qas_per_slice: int = 5,
    max_docs: int | None = None,   # for testing; set None for all
):
    with out_path.open("w", encoding="utf-8") as out_f:
        for i, path in enumerate(sorted(chunk_dir.glob("*.rag.chunks.json"))):
            if max_docs is not None and i >= max_docs:
                break

            chunks = load_chunks(path)
            slices = make_slices_from_chunks(chunks, max_chars=3000, min_chars=500)

            for slice_idx, s in enumerate(slices):
                qas = generate_qas_self_instruct(
                    doc_id=s["doc_id"],
                    text=s["text"],
                    n_qas=n_qas_per_slice,
                )

                for qa_idx, qa in enumerate(qas):
                    record = {
                        # identifiers
                        "id": f"{s['doc_id']}_slice{slice_idx}_qa{qa_idx}",
                        "dataset": DATASET_NAME,
                        "doc_id": s["doc_id"],
                        "slice_id": s["slice_id"],
                        "slice_index": slice_idx,
                        "chunk_ids": s["chunk_ids"],

                        # QA
                        "question": qa["question"],
                        "answer": qa["answer"],

                        # optional, but useful for debugging / later analysis
                        "source_text": s["text"],  # or s["text"][:2000] if size is an issue

                        # meta
                        "generator_model": MODEL,  # your OpenAI model name, e.g. "gpt-4.1-mini"
                        "created_at": datetime.utcnow().isoformat(),
                    }
                    out_f.write(json.dumps(record) + "\n")

            print(f"Processed {path.name}: {len(slices)} slices")

# run a small test first
build_eval_qas_jsonl(max_docs=3)  # try on 3 papers first


Processed 2001.08361.rag.chunks.json: 25 slices
Processed 2005.03141.rag.chunks.json: 9 slices
Processed 2101.03961.rag.chunks.json: 31 slices


# Self-ask + Filter

In [43]:
VERIFY_PROMPT = """
You are checking if a question-answer pair is fully supported by the given text.

TEXT:
\"\"\"{text}\"\"\"

QUESTION: {question}
ANSWER: {answer}

Does the text above provide enough information to fully support this answer, without relying on outside knowledge?

Respond with a single word: "YES" or "NO".
"""

In [44]:
# You can later swap get_response here to a cheaper/smaller model if you want.

def verify_qa_with_gpt(text: str, question: str, answer: str) -> bool:
    prompt = VERIFY_PROMPT.format(text=text, question=question, answer=answer)
    resp = get_response(prompt).strip().upper()
    # be a bit forgiving
    if "YES" in resp and "NO" not in resp:
        return True
    if "NO" in resp and "YES" not in resp:
        return False
    # fallback: treat anything weird as NO
    return False

In [45]:
# lets use a smaller model for filtering wrong QA pairs

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# use Phi-4-mini-instruct as judge model
JUDGE_MODEL_ID = "microsoft/Phi-4-mini-instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.float16,   # or bfloat16 / float32 if needed
    device_map="auto",           # or remove this if you want pure CPU
)

def filter_response(prompt: str) -> str:
    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)
    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.0,
        pad_token_id=judge_tokenizer.eos_token_id,
    )
    gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
    return judge_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()

def verify_qa_with_phi(text: str, question: str, answer: str) -> bool:
    prompt = VERIFY_PROMPT.format(text=text, question=question, answer=answer)
    resp = filter_response(prompt).strip().upper()
    if "YES" in resp and "NO" not in resp:
        return True
    if "NO" in resp and "YES" not in resp:
        return False
    return False


Loading checkpoint shards: 100%|██████████| 2/2 [00:37<00:00, 18.86s/it]


In [46]:
def generate_qas_self_instruct_filtered(doc_id: str,
                                        text: str,
                                        n_qas: int = 5,
                                        overgenerate_factor: int = 2,
                                        ) -> List[QA]:
    # 1) generate more than we need
    raw_qas = generate_qas_self_instruct(
        doc_id=doc_id,
        text=text,
        n_qas=n_qas * overgenerate_factor,
    )

    # 2) filter using the verifier
    filtered: List[QA] = []
    for qa in raw_qas:
        if verify_qa_with_phi(text, qa["question"], qa["answer"]):
            filtered.append(qa)
        if len(filtered) >= n_qas:
            break

    # you might end up with fewer than n_qas if many are rejected
    return filtered

In [49]:
slice0 = slices[0]

print("\nFiltered QAs:")
filtered_qas = generate_qas_self_instruct_filtered(slice0["doc_id"], slice0["text"], n_qas=5)
for qa in filtered_qas:
    print("Q:", qa["question"])
    print("A:", qa["answer"])
    print("----")


Filtered QAs:
Q: What is speculative execution in the context of processors?
A: Speculative execution is an optimization technique where a task is performed in parallel to verifying if it's actually needed, increasing concurrency.
----
Q: How does the paper generalize speculative execution?
A: The paper generalizes speculative execution to the stochastic setting, where a task might be needed with some probability.
----
Q: What is the novel sampling method introduced in this work?
A: The novel sampling method introduced is called speculative sampling.
----
Q: Which models are used as the target and approximation models in speculative decoding?
A: The target model is denoted as M_p, and the more efficient approximation model is denoted as M_q.
----
Q: How does speculative decoding accelerate sampling from autoregressive models?
A: By using M_q to generate multiple completions in parallel and evaluating these guesses with M_p, accepting those that maintain the same distribution, thus red

# LangChain Synthetic QA

In [ ]:
from langchain.evaluation.qa import QAGenerateChain  # or synthetic modules depending on version

qa_generator = QAGenerateChain.from_llm(your_llm)

In [ ]:
def generate_qas_langchain(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    res = qa_generator.generate(
        examples=[{"doc": text}],
        n=n_qas,  # depends on exact signature
    )
    # res might be a list of dicts like {"question": ..., "answer": ...}
    qas = []
    for qa in res:
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas

# DeepEval synthetic module

In [ ]:
from deepeval.synthetic import generate_qa_dataset  # name illustrative

def generate_qas_deepeval(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    dataset = generate_qa_dataset(
        documents=[{"id": doc_id, "text": text}],
        num_questions_per_doc=n_qas,
        # plus any config like LLM, style, etc.
    )
    # dataset might be a list of {"question": ..., "answer": ..., "metadata": ...}
    qas = []
    for qa in dataset:
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas